# Transcribe Videos from Urdu to English

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

In [ ]:
the_folder_you_have_files_in=""
video_folder= "/content/drive/MyDrive/{the_folder_you_have_files_in}"
output_folder = video_folder

In [ ]:
!pip install -q faster-whisper
!apt-get -y install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "medium",          # change to "small" if you want faster
    device="cuda",
    compute_type="float16"
)

# Transcribing process

In [ ]:
import subprocess

video_files = [
    f for f in os.listdir(video_folder)
    if f.endswith((".mp4", ".mkv", ".mov", ".avi"))
]

print("Found videos:", video_files)

for video in video_files:
    video_path = os.path.join(video_folder, video)
    audio_path = video_path + ".wav"

    print("\n==============================")
    print("Processing:", video)
    print("==============================")

    # STEP 1: Extract audio
    subprocess.run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-ar", "16000",
        "-ac", "1",
        audio_path
    ])

    # STEP 2: Transcribe + translate Urdu → English
    segments, info = model.transcribe(
        audio_path,
        task="translate"
    )

    # STEP 3: Save output
    output_text = ""

    for seg in segments:
        line = f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}"
        print(line)
        output_text += line + "\n"

    output_file = os.path.join(output_folder, video + "_english.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(output_text)

    print("\nSaved to:", output_file)

Found videos: ['chunk_00.mp4', 'chunk_01.mp4', 'chunk_02.mp4', 'chunk_03.mp4', 'chunk_04.mp4', 'chunk_05.mp4', 'chunk_06.mp4', 'chunk_07.mp4', 'chunk_08.mp4', 'chunk_09.mp4']

Processing: chunk_00.mp4
[0.00s -> 2.00s]  This is Power 2.
[2.00s -> 4.00s]  As we did in 1st.
[26.00s -> 28.00s]  Qualify or Disqualify
[28.00s -> 32.00s]  There will be 6 to 20 attorneys in this law firm.
[32.00s -> 36.00s]  Out of them, we have to give only 2 attorneys.
[36.00s -> 40.00s]  And we have to highlight that lead.
[40.00s -> 42.00s]  Like I will take data from this.
[42.00s -> 44.00s]  So I will show you practically.
[44.00s -> 50.00s]  And in the qualify criteria, there should be a law firm.
[50.00s -> 52.00s]  There should be no other firm.
[52.00s -> 56.00s]  Like this is the practice area.
[56.00s -> 58.00s]  You can see.
[58.00s -> 60.00s]  Like this.
[60.00s -> 66.00s]  And in the disqualify criteria, we don't have to take off-concert.
[66.00s -> 70.00s]  If the email is not found, then it is

Whisper flow often generates errors for audios above 12 minutes, In this example we are making the video into chunks of 10 parts

In [ ]:
!pip install -q faster-whisper
!apt-get -y install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
# =========================
# 1. SET PATHS
# =========================
import os
import subprocess

video_folder = "/content/drive/MyDrive/Phoneburner"

video_file = [f for f in os.listdir(video_folder) if f.endswith((".mp4", ".mkv", ".mov"))][0]
video_path = os.path.join(video_folder, video_file)

chunks_dir = os.path.join(video_folder, "chunks")
output_dir = os.path.join(video_folder, "transcripts")

os.makedirs(chunks_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("Video found:", video_file)


# =========================
# 2. GET VIDEO DURATION
# =========================
result = subprocess.run([
    "ffprobe", "-v", "error",
    "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1",
    video_path
], stdout=subprocess.PIPE, text=True)

duration = float(result.stdout.strip())
segment_time = duration / 10

print("Duration:", duration)
print("Each chunk seconds:", segment_time)


# =========================
# 3. SPLIT VIDEO INTO 10 PARTS
# =========================
chunk_pattern = os.path.join(chunks_dir, "chunk_%02d.mp4")

subprocess.run([
    "ffmpeg", "-y",
    "-i", video_path,
    "-c", "copy",
    "-map", "0",
    "-f", "segment",
    "-segment_time", str(segment_time),
    "-reset_timestamps", "1",
    chunk_pattern
])


After dividing into chunks you can goto Transcribe step